# SPARQL Inferencing — User Guide

Inferencing is the process of deriving new information from existing graph information according to a specified set of rules, semantics, or expressions.  Inferencing creates new triples based on existing triples in the graph. 

As described in the [Graph Inferencing guide](02b-graphs-inferencing.ipynb), a reasoning *profile* can be applied to a graph directly via `infer()`. The resultant graph can then be queried as described in the [SPARQL guide](03-sparql.ipynb). 

Alternatively, `StarLayerGraph.query()` lets a query itself specify the *entailment regime* to apply via `entailment=`. This lets you query a graph with the benefit of inferencing, without first running inference on the underlying graph. 

A `StarLayerGraph` query with no `entailment` uses **Simple Entailment** — plain graph pattern matching. The SPARQL 1.2 [Entailment Regimes](https://www.w3.org/TR/sparql12-entailment/) specification defines six such regimes beyond that baseline.  StarLayerGraph provides partial support for entailment regimes as outlined below. 

| # | Regime | Status |
|---|---|---|
| 1 | RDF Entailment | ✅ Done — `entailment="rdf"` (section 1) |
| 2 | RDFS Entailment | ✅ Done — `entailment="rdfs"` (section 2) |
| 3 | D-Entailment (datatype canonicalization) | ❌ Not supported |
| 4 | OWL 2 RDF-Based Semantics | ✅ Done for OWL 2 RL, the decidable rule-based fragment — `entailment="owl-rl"` (section 3)<br>❌ Not supported for the full regime — undecidable in general |
| 5 | OWL 2 Direct Semantics | 🔜 Planned — pending a DL reasoner (HermiT-class) |
| 6 | RIF Core Entailment | ❌ Not supported |



## How to run this notebook

See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet. This guide assumes StarLayer has been pip installed.

Run cells from top to bottom — later sections may reuse variables from earlier sections.

In [1]:
from starlayergraph import StarLayerGraph, Namespace, RDF

EX = Namespace("http://example.org/")

## 1. `entailment="rdf"` — RDF Entailment

A narrower, weaker entailment regime than `entailment="rdfs"` (section 2) — included for completeness with the SPARQL specification's regime list. `entailment="rdf"` covers one rule (rdfD2): every term used anywhere as a predicate is entailed to be `rdf:type rdf:Property`. Useful for schema introspection ("what properties does this data actually use?"). In practice `entailment="rdfs"`/`"owl-rl"` are a more useful choice for querying.

In [2]:
g4 = StarLayerGraph()
g4.bind("ex", EX)
g4.parse(data='''
    @prefix ex: <http://example.org/> .
    ex:alice ex:worksAt ex:Acme .
''', format="turtle12")

QUERY_RDF = "PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> SELECT ?p WHERE { ?p a rdf:Property }"

print("rdf:Property in the graph:")
print("plain (no entailment):", [str(r.p) for r in g4.query(QUERY_RDF)])
print("entailment=\"rdf\":       ", [str(r.p) for r in g4.query(QUERY_RDF, entailment="rdf")])

rdf:Property in the graph:
plain (no entailment): []
entailment="rdf":        ['http://example.org/worksAt']


## 2. `entailment="rdfs"` — RDFS Entailment

A broader entailment regime than `entailment="rdf"` (section 1) — `entailment="rdfs"` covers the full RDFS "data" ruleset, including four separate rules.

- **`rdfs:subClassOf`** (rdfs9) — a class's declared superclasses apply to `rdf:type` queries across any number of hops.
- **`rdfs:domain`** (rdfs2) — a property's declared domain entails a type for its subject.
- **`rdfs:range`** (rdfs3) — a property's declared range entails a type for its object.
- **`rdfs:subPropertyOf`** (rdfs5/rdfs7) — itself reflexive and transitive, so a query for a general relation also matches its more specific sub-relations.

`entailment="rdfs"` deliberately excludes RDFS's remaining "vocabulary-level" rules — universal `rdfs:Resource` typing (rdfs4a/4b) and the fixed RDFS axiomatic triples — since they don't answer a real question about the data. 

`entailment="rdfs"` queries are handled through a query-time rewrite of the SPARQL query against the current graph data. No copy or mutation of the original graph is required.

In [3]:
# subclass entailment.  Alice is an employee entails from the fact she is a manager.
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:Employee rdfs:subClassOf ex:Person .
    ex:alice a ex:Manager .
''', format="turtle12")

QUERY = "PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Employee }"

print("plain (no entailment):", [str(r.x) for r in g.query(QUERY)])
print("entailment=\"rdfs\":       ", [str(r.x) for r in g.query(QUERY, entailment="rdfs")])
print("triple count unchanged - no data was copied or added:", len(g))

plain (no entailment): []
entailment="rdfs":        ['http://example.org/alice']
triple count unchanged - no data was copied or added: 3


### 2.a Reflexivity and multi-hop transitivity

`rdfs:subClassOf` entailment is both reflexive (a direct type still matches itself) and transitive across any number of hops. 

In [4]:
# Alice is a person is entailed from the fact that she is an employee which is entailed from the fact she is a manager.
rows = g.query("PREFIX ex: <http://example.org/> SELECT ?type WHERE { ex:alice a ?type }", entailment="rdfs")
print("every entailed type:", sorted(str(r.type) for r in rows))

rows = g.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Person }", entailment="rdfs")
print("two hops away (Manager -> Employee -> Person):", [str(r.x) for r in rows])

every entailed type: ['http://example.org/Employee', 'http://example.org/Manager', 'http://example.org/Person']
two hops away (Manager -> Employee -> Person): ['http://example.org/alice']


### 2.c `rdfs:domain`, `rdfs:range`, and `rdfs:subPropertyOf`

The same `entailment="rdfs"` value also covers the rest of the RDFS "data" ruleset — a property's declared `rdfs:domain`/`rdfs:range` entails a type for its subject/object, and `rdfs:subPropertyOf` lets a query for a general relation also match its more specific sub-relations. 

In [5]:
# If we know that Alice works at Acme that entails Alice is a person and Acme is an organization.
# If we know that Alice has a parent bob, that entails that Alice has a relative bob.
g2 = StarLayerGraph()
g2.bind("ex", EX)
g2.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:worksAt rdfs:domain ex:Person ;
               rdfs:range ex:Organization .
    ex:hasParent rdfs:subPropertyOf ex:hasRelative .
    ex:alice ex:worksAt ex:Acme .
    ex:alice ex:hasParent ex:bob .
''', format="turtle12")

rows = g2.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Person }", entailment="rdfs")
print("alice is a Person (via rdfs:domain):     ", [str(r.x) for r in rows])

rows = g2.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Organization }", entailment="rdfs")
print("Acme is an Organization (via rdfs:range):", [str(r.x) for r in rows])

rows = g2.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ex:alice ex:hasRelative ?x }", entailment="rdfs")
print("alice hasRelative bob (via subPropertyOf):", [str(r.x) for r in rows])

alice is a Person (via rdfs:domain):      ['http://example.org/alice']
Acme is an Organization (via rdfs:range): ['http://example.org/Acme']
alice hasRelative bob (via subPropertyOf): ['http://example.org/bob']


## 3. `entailment="owl-rl"` — OWL 2 RDF-Based Semantics

`entailment="owl-rl"` is a broader entailment regime than `entailment="rdfs"` (section 2).

In addition to RDFS's four "data" rules, `entailment="owl-rl"` also covers genuine OWL constructs:
- `owl:equivalentClass`/`owl:equivalentProperty` (treating declared-equivalent terms interchangeably)
- `owl:TransitiveProperty`/`owl:SymmetricProperty`/`owl:inverseOf` (inference over a property's own declared characteristics)
- `owl:sameAs` (congruence closure — two individuals declared the same share every other fact about them)

OWL 2 RL's rule set is a superset of RDFS's, so `entailment="owl-rl"` alone already includes everything `entailment="rdfs"` does too.

It does **not** cover the full OWL 2 RDF-Based Semantics regime the SPARQL spec defines — only its OWL 2 RL fragment. Full OWL is undecidable in general (it has roughly first-order expressive power, plus some of RDF's own reflective quirks); `owlrl`, the Python library this rests on, implements specifically the RL profile because that's the one version of OWL with a terminating, rule-based algorithm.

### 3.a `owl:equivalentClass` and `owl:equivalentProperty`

Declaring two classes (or two properties) equivalent lets an instance of one be recognized as the other.

In [6]:
# Alice is a team lead is entailed from Alice is a manager, because team lead and manager are the same.
g_owl = StarLayerGraph()
g_owl.bind("ex", EX)
g_owl.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .
''', format="turtle12")

QUERY_OWL = "PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:TeamLead }"
print("entailment=\"rdfs\"   (misses owl:equivalentClass):", [str(r.x) for r in g_owl.query(QUERY_OWL, entailment="rdfs")])
print("entailment=\"owl-rl\" (catches it):                ", [str(r.x) for r in g_owl.query(QUERY_OWL, entailment="owl-rl")])

entailment="rdfs"   (misses owl:equivalentClass): []
entailment="owl-rl" (catches it):                 ['http://example.org/alice']


### 3.b Property characteristics: `owl:TransitiveProperty`, `owl:SymmetricProperty`, `owl:inverseOf`

A property's own declared characteristics entail new facts about how it's used — transitive chains, symmetric reversal, and inverse-property pairing. 

In [7]:
# SalesTeam is part of AcmeCorp is entailed via the transitive property.
# bob is friendOf alice is entailed via the symmetric property.
# dave is managedBy carol is entailed via the inverse property.
g_props = StarLayerGraph()
g_props.bind("ex", EX)
g_props.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .

    ex:partOf a owl:TransitiveProperty .
    ex:SalesTeam ex:partOf ex:SalesDept .
    ex:SalesDept ex:partOf ex:AcmeCorp .

    ex:friendOf a owl:SymmetricProperty .
    ex:alice ex:friendOf ex:bob .

    ex:manages owl:inverseOf ex:managedBy .
    ex:carol ex:manages ex:dave .
''', format="turtle12")

rows = g_props.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ex:SalesTeam ex:partOf ?x }", entailment="owl-rl")
print("SalesTeam transitively part of :", sorted(str(r.x) for r in rows))

rows = g_props.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ex:bob ex:friendOf ?x }", entailment="owl-rl")
print("bob friendOf (symmetric)       :", [str(r.x) for r in rows])

rows = g_props.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ex:dave ex:managedBy ?x }", entailment="owl-rl")
print("dave managedBy (via inverseOf) :", [str(r.x) for r in rows])

SalesTeam transitively part of : ['http://example.org/AcmeCorp', 'http://example.org/SalesDept']
bob friendOf (symmetric)       : ['http://example.org/alice']
dave managedBy (via inverseOf) : ['http://example.org/carol']


### 3.c `owl:sameAs` — congruence closure

Declaring two individuals `owl:sameAs` each other means anything asserted about one is entailed about the other too — congruence closure.

In [8]:
#Alice Smith works at Acme, because Alice Smith and Alice Jones are the same entity.
g_same = StarLayerGraph()
g_same.bind("ex", EX)
g_same.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:aliceSmith owl:sameAs ex:aliceJones .
    ex:aliceJones ex:worksAt ex:Acme .
''', format="turtle12")

QUERY_SAME = "PREFIX ex: <http://example.org/> SELECT ?org WHERE { ex:aliceSmith ex:worksAt ?org }"
print("plain (no entailment):          ", [str(r.org) for r in g_same.query(QUERY_SAME)])
print("entailment=\"owl-rl\" (via sameAs):", [str(r.org) for r in g_same.query(QUERY_SAME, entailment="owl-rl")])

plain (no entailment):           []
entailment="owl-rl" (via sameAs): ['http://example.org/Acme']


### 3.d Caching via `infer=`

Unlike `entailment="rdfs"`, `entailment="owl-rl"` requires a potentially time-consuming calcualtion of the entailed graph.  

By default (`infer="changed"`), `entailment="owl-rl"` will cache the entailed graph and reuse it for subserquent queries unless the underlying graph changes.  The other choices are `infer="cached"` which will forve the re-use of the cached entailed graph regardless of changes and  `infer="always"` will force a re-compute of the entailed graph.  

In [9]:
g_owl.add((EX.carol, RDF.type, EX.Manager))
rows = g_owl.query(QUERY_OWL, entailment="owl-rl")
print("default (changed), picks up carol:", sorted(str(r.x) for r in rows))

g_owl.add((EX.dave, RDF.type, EX.Manager))
stale = g_owl.query(QUERY_OWL, entailment="owl-rl", infer="cached")
print("infer=\"cached\" (misses dave - stale on purpose):", sorted(str(r.x) for r in stale))

fresh = g_owl.query(QUERY_OWL, entailment="owl-rl", infer="always")
print("infer=\"always\" (picks up dave):                 ", sorted(str(r.x) for r in fresh))

default (changed), picks up carol: ['http://example.org/alice', 'http://example.org/carol']
infer="cached" (misses dave - stale on purpose): ['http://example.org/alice', 'http://example.org/carol']
infer="always" (picks up dave):                  ['http://example.org/alice', 'http://example.org/carol', 'http://example.org/dave']


### 3.e Detecting inconsistencies

`entailment="owl-rl"` inherits `owlrl`'s consistency checks (see the Inferencing guide's [section 2.4](02b-graphs-inferencing.ipynb#2.4-Detecting-inconsistencies)). When querying using `owl-rl` entailment, a result will be returned even if the underlying inference detects an inconsistency. Running a simple `ASK` query can detect the inconsistency:

In [10]:
ASK_INCONSISTENT = """
PREFIX err: <http://www.daml.org/2002/03/agents/agent-ont#>
ASK { ?e a err:ErrorMessage }
"""

# Dog and Cat are declared disjoint, but fido is asserted to be both
g_bad = StarLayerGraph()
g_bad.bind("ex", EX)
g_bad.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Dog owl:disjointWith ex:Cat .
    ex:fido a ex:Dog, ex:Cat .
''', format="turtle12")

print("g_bad (genuine inconsistency):  ", bool(g_bad.query(ASK_INCONSISTENT, entailment="owl-rl")))

# a normal query still returns a result - nothing is retracted, the inconsistency isn't surfaced here
rows = g_bad.query("PREFIX ex: <http://example.org/> SELECT ?c WHERE { ex:fido a ?c }", entailment="owl-rl")
print("fido's classes (still returned despite the inconsistency):", sorted(str(r.c) for r in rows))

g_bad (genuine inconsistency):   True
fido's classes (still returned despite the inconsistency): ['http://example.org/Cat', 'http://example.org/Dog']


## 4. Which entailment regime to use

- **`entailment="rdf"`** (section 1) — narrow, and rarely used directly.
- **`entailment="rdfs"`** (section 2) — Cheapest per query. Covers the full RDFS "data" ruleset. The right default whenever your data doesn't need genuine OWL constructs.
- **`entailment="owl-rl"`** section 3, for graphs using OWL characteristics that RDFS rewriting can't express. 


## Further Reading

1. **[Getting Started](01-getting-started.ipynb)** — install, first parse, first query, first validate.
2. **[Graphs](02-graphs.ipynb)** — `TripleTerm`/`DirLangString` semantics, Turtle 1.2 reification syntax.
   - 2.b **[Inferencing](02b-graphs-inferencing.ipynb)** — RDFS/OWL-RL reasoning via `owlrl`, including `StarLayerGraph.infer()`.
3. **SPARQL**
   - 3.a **[SPARQL rules (pending)](03a-sparql-rules-pending.md)** — SPARQL-RL (SRL), a separate Datalog-style rules language, deliberately out of scope for this project.
   - 3.b **SPARQL inferencing** — this guide, covering `entailment="rdf"`/`"rdfs"`/`"owl-rl"`.
5. **Other**
   - 5.b **[Working with backend graph databases](05b-backend-graph-databases.ipynb)** — `entailment="native"`, delegating reasoning to a Fuseki/Jena backend instead of `entailment="rdf"`/`"rdfs"`/`"owl-rl"`.
   - 5.d **[Canonical hashing and graph comparison](05d-canonical-hashing.ipynb)** — RDFC-1.0 canonicalization/hashing and graph isomorphism.
   - 5.e. hermit reasoning.